In [2]:
import cv2
import numpy as np
import json
import hashlib
import logging
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Tuple, Optional, Dict, Any

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_DIR = Path(r"C:\Users\USER\Desktop\GRAD_WORk\runic_images_cleaned\VAL_DATASET")
OUTPUT_DIR = Path(r"C:\Users\USER\Desktop\GRAD_WORk\runic_images_cleaned\NORMALIZED_VAL_DATASET")
DEBUG_DIR = OUTPUT_DIR / "debug"
METADATA_DIR = OUTPUT_DIR / "metadata"

TARGET_SHORT_SIDE = 640
DEBUG_MODE = True
MIN_IMAGE_SIZE = 200
MAX_ASPECT_RATIO = 10
MIN_BLUR_THRESHOLD = 50

# ============================================================================
# UNICODE PATH FIX FOR OPENCV
# ============================================================================

def imread_unicode(filepath: Path) -> Optional[np.ndarray]:
    """Чтение изображения с поддержкой Unicode путей"""
    try:
        with open(filepath, 'rb') as f:
            file_bytes = np.asarray(bytearray(f.read()), dtype=np.uint8)
        img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
        return img
    except Exception as e:
        logger.error(f"Failed to read {filepath.name}: {e}")
        return None


def imwrite_unicode(filepath: Path, img: np.ndarray) -> bool:
    """Запись изображения с поддержкой Unicode путей"""
    try:
        is_success, buffer = cv2.imencode('.png', img)
        if is_success:
            with open(filepath, 'wb') as f:
                f.write(buffer)
            return True
        return False
    except Exception as e:
        logger.error(f"Failed to write {filepath.name}: {e}")
        return False


# ============================================================================
# SETUP
# ============================================================================

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
if DEBUG_MODE:
    DEBUG_DIR.mkdir(exist_ok=True, parents=True)
METADATA_DIR.mkdir(exist_ok=True, parents=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(OUTPUT_DIR / 'processing.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# DATA CLASSES
# ============================================================================

@dataclass
class ProcessingStats:
    """Статистика обработки датасета"""
    total: int = 0
    success: int = 0
    skipped_not_readable: int = 0
    skipped_too_small: int = 0
    skipped_bad_aspect: int = 0
    skipped_too_blurry: int = 0
    skipped_no_contours: int = 0
    skipped_rotation_failed: int = 0
    
    def print_report(self):
        """Вывод финального отчёта"""
        logger.info("=" * 60)
        logger.info("PROCESSING REPORT")
        logger.info("=" * 60)
        logger.info(f"Total images:           {self.total}")
        logger.info(f"Successfully processed: {self.success}")
        logger.info(f"Failed to read:         {self.skipped_not_readable}")
        logger.info(f"Too small:              {self.skipped_too_small}")
        logger.info(f"Bad aspect ratio:       {self.skipped_bad_aspect}")
        logger.info(f"Too blurry:             {self.skipped_too_blurry}")
        logger.info(f"No contours found:      {self.skipped_no_contours}")
        logger.info(f"Rotation failed:        {self.skipped_rotation_failed}")
        logger.info("=" * 60)
        if self.total > 0:
            success_rate = (self.success / self.total) * 100
            logger.info(f"Success rate: {success_rate:.1f}%")


@dataclass
class ImageMetadata:
    """Метаданные обработанного изображения"""
    original_filename: str
    original_size: Tuple[int, int]
    normalized_size: Tuple[int, int]
    rotation_angle: float
    scale_factor: float
    contour_area: int
    bbox: Tuple[int, int, int, int]
    image_hash: str
    processing_date: str
    clahe_applied: bool = True
    
    def to_dict(self) -> Dict[str, Any]:
        """Конвертация в словарь для JSON с приведением numpy типов"""
        def convert_value(v):
            """Конвертация numpy типов в Python типы"""
            if isinstance(v, (np.integer, np.int32, np.int64)):
                return int(v)
            elif isinstance(v, (np.floating, np.float32, np.float64)):
                return float(v)
            elif isinstance(v, np.ndarray):
                return v.tolist()
            elif isinstance(v, tuple):
                return tuple(convert_value(x) for x in v)
            return v
        
        data = asdict(self)
        return {k: convert_value(v) for k, v in data.items()}


# ============================================================================
# QUALITY CHECKS
# ============================================================================

def compute_image_hash(img: np.ndarray) -> str:
    """Вычисление SHA256 хеша изображения"""
    return hashlib.sha256(img.tobytes()).hexdigest()[:16]


def is_suitable_for_ocr(img: np.ndarray) -> Tuple[bool, str]:
    """Проверка качества изображения перед обработкой"""
    h, w = img.shape[:2]
    
    if min(h, w) < MIN_IMAGE_SIZE:
        return False, f"too_small ({min(h, w)} < {MIN_IMAGE_SIZE})"
    
    aspect_ratio = max(h, w) / min(h, w)
    if aspect_ratio > MAX_ASPECT_RATIO:
        return False, f"bad_aspect_ratio ({aspect_ratio:.1f} > {MAX_ASPECT_RATIO})"
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var < MIN_BLUR_THRESHOLD:
        return False, f"too_blurry (variance={laplacian_var:.1f})"
    
    return True, "ok"


# ============================================================================
# IMAGE NORMALIZATION
# ============================================================================

def normalize_image(
    img: np.ndarray, 
    img_name: str
) -> Tuple[np.ndarray, ImageMetadata]:
    """Нормализация изображения рунической надписи"""
    original_size = (img.shape[0], img.shape[1])
    debug_path = DEBUG_DIR / img_name if DEBUG_MODE else None
    
    if debug_path:
        debug_path.mkdir(exist_ok=True)
    
    # 1. GRAYSCALE
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if debug_path:
        imwrite_unicode(debug_path / "01_gray.png", gray)
    
    # 2. CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    if debug_path:
        imwrite_unicode(debug_path / "02_clahe.png", enhanced)
    
    # 3. THRESHOLD
    _, binary = cv2.threshold(
        enhanced, 0, 255, 
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    if debug_path:
        imwrite_unicode(debug_path / "03_binary.png", binary)
    
    # 4. MORPHOLOGY
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    morphed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    if debug_path:
        imwrite_unicode(debug_path / "04_morphology.png", morphed)
    
    # 5. FIND LARGEST CONTOUR
    contours, _ = cv2.findContours(
        morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    
    if not contours:
        raise ValueError("no_contours_found")
    
    largest_contour = max(contours, key=cv2.contourArea)
    contour_area = int(cv2.contourArea(largest_contour))
    
    rect = cv2.minAreaRect(largest_contour)
    box = cv2.boxPoints(rect)
    box = np.intp(box)
    
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    if debug_path:
        vis = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
        cv2.drawContours(vis, [largest_contour], -1, (0, 255, 0), 2)
        cv2.drawContours(vis, [box], -1, (0, 0, 255), 2)
        imwrite_unicode(debug_path / "05_contours.png", vis)
    
    cropped = enhanced[y:y+h, x:x+w]
    
    if debug_path:
        imwrite_unicode(debug_path / "06_cropped.png", cropped)
    
    # 6. ORIENTATION CORRECTION
    coords = np.column_stack(np.where(cropped < 128))
    rotation_angle = 0.0
    
    if len(coords) > 10:
        try:
            mean, eigenvectors = cv2.PCACompute(
                coords.astype(np.float32), 
                mean=None
            )
            angle = np.arctan2(eigenvectors[0, 0], eigenvectors[0, 1])
            rotation_angle = float(np.degrees(angle))  # Явная конвертация в float
            
            if abs(rotation_angle) > 5:
                center = (cropped.shape[1] // 2, cropped.shape[0] // 2)
                rot_mat = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)
                
                cos = np.abs(rot_mat[0, 0])
                sin = np.abs(rot_mat[0, 1])
                new_w = int((cropped.shape[0] * sin) + (cropped.shape[1] * cos))
                new_h = int((cropped.shape[0] * cos) + (cropped.shape[1] * sin))
                
                rot_mat[0, 2] += (new_w / 2) - center[0]
                rot_mat[1, 2] += (new_h / 2) - center[1]
                
                cropped = cv2.warpAffine(
                    cropped, rot_mat, (new_w, new_h),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_REPLICATE
                )
                
                if debug_path:
                    imwrite_unicode(debug_path / "07_rotated.png", cropped)
        except Exception as e:
            logger.warning(f"{img_name}: rotation failed ({e}), skipping rotation")
            rotation_angle = 0.0
    
    # 7. RESIZE
    crop_h, crop_w = cropped.shape
    scale = float(TARGET_SHORT_SIDE / min(crop_h, crop_w))  # Явная конвертация в float
    new_w, new_h = int(crop_w * scale), int(crop_h * scale)
    
    resized = cv2.resize(
        cropped, (new_w, new_h), 
        interpolation=cv2.INTER_LINEAR
    )
    
    if debug_path:
        imwrite_unicode(debug_path / "08_final.png", resized)
    
    metadata = ImageMetadata(
        original_filename=img_name,
        original_size=original_size,
        normalized_size=(new_h, new_w),
        rotation_angle=rotation_angle,
        scale_factor=scale,
        contour_area=contour_area,
        bbox=(int(x), int(y), int(w), int(h)),  # Явная конвертация в int
        image_hash=compute_image_hash(resized),
        processing_date=datetime.now().isoformat()
    )
    
    return resized, metadata


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Главная функция обработки датасета"""
    logger.info("Starting image normalization pipeline")
    logger.info(f"Input directory:  {INPUT_DIR}")
    logger.info(f"Output directory: {OUTPUT_DIR}")
    logger.info(f"Debug mode:       {DEBUG_MODE}")
    logger.info("=" * 60)
    
    stats = ProcessingStats()
    supported_formats = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    
    image_files = [
        f for f in INPUT_DIR.iterdir() 
        if f.suffix.lower() in supported_formats
    ]
    
    logger.info(f"Found {len(image_files)} image files")
    
    for img_path in image_files:
        stats.total += 1
        logger.info(f"[{stats.total}/{len(image_files)}] Processing {img_path.name}")
        
        img = imread_unicode(img_path)
        if img is None:
            logger.warning(f"  ✗ Cannot read file")
            stats.skipped_not_readable += 1
            continue
        
        is_ok, reason = is_suitable_for_ocr(img)
        if not is_ok:
            logger.warning(f"  ✗ Skipped: {reason}")
            if "too_small" in reason:
                stats.skipped_too_small += 1
            elif "aspect" in reason:
                stats.skipped_bad_aspect += 1
            elif "blurry" in reason:
                stats.skipped_too_blurry += 1
            continue
        
        try:
            normalized, metadata = normalize_image(img, img_path.stem)
        except ValueError as e:
            error_msg = str(e)
            logger.error(f"  ✗ Processing failed: {error_msg}")
            if "no_contours" in error_msg:
                stats.skipped_no_contours += 1
            else:
                stats.skipped_rotation_failed += 1
            continue
        except Exception as e:
            logger.error(f"  ✗ Unexpected error: {e}")
            stats.skipped_rotation_failed += 1
            continue
        
        out_path = OUTPUT_DIR / f"{img_path.stem}.png"
        success = imwrite_unicode(out_path, normalized)
        
        if not success:
            logger.error(f"  ✗ Failed to write {out_path}")
            continue
        
        metadata_path = METADATA_DIR / f"{img_path.stem}.json"
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(metadata.to_dict(), f, indent=2, ensure_ascii=False)
        
        stats.success += 1
        logger.info(
            f"  ✓ Success: {metadata.normalized_size[1]}x{metadata.normalized_size[0]} "
            f"(rotated {metadata.rotation_angle:.1f}°, scaled {metadata.scale_factor:.2f}x)"
        )
    
    stats.print_report()
    
    stats_path = OUTPUT_DIR / "processing_stats.json"
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(asdict(stats), f, indent=2)
    
    logger.info(f"Statistics saved to {stats_path}")
    logger.info("Processing complete!")


if __name__ == "__main__":
    main()

2026-02-05 11:23:52,050 - INFO - Starting image normalization pipeline
2026-02-05 11:23:52,051 - INFO - Input directory:  C:\Users\USER\Desktop\GRAD_WORk\runic_images_cleaned\VAL_DATASET
2026-02-05 11:23:52,052 - INFO - Output directory: C:\Users\USER\Desktop\GRAD_WORk\runic_images_cleaned\NORMALIZED_VAL_DATASET
2026-02-05 11:23:52,052 - INFO - Debug mode:       True
2026-02-05 11:23:52,053 - INFO - ============================================================
2026-02-05 11:23:52,055 - INFO - Found 471 image files
2026-02-05 11:23:52,056 - INFO - [1/471] Processing 10_5.jpg
2026-02-05 11:23:53,331 - INFO -   ✓ Success: 1079x640 (rotated 5.5°, scaled 0.23x)
2026-02-05 11:23:53,332 - INFO - [2/471] Processing 10_5_1.jpg
2026-02-05 11:23:53,387 - INFO -   ✓ Success: 1220x640 (rotated 2.9°, scaled 1.43x)
2026-02-05 11:23:53,388 - INFO - [3/471] Processing 10_5_3.jpg
2026-02-05 11:23:53,423 - INFO -   ✓ Success: 959x640 (rotated 3.2°, scaled 1.58x)
2026-02-05 11:23:53,424 - INFO - [4/471] Pr